In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
admissions = pd.read_csv("../data/processed/admissions_clean.csv")
bed_capacity = pd.read_csv("../data/raw/bed_capacity.csv")
departments = pd.read_csv("../data/processed/departments_clean.csv")

print("Admissions shape:", admissions.shape)
print("Bed capacity shape:", bed_capacity.shape)

print("\nAdmissions columns:")
print(admissions.columns.tolist())

print("\nBed capacity:")
display(bed_capacity.head())

Admissions shape: (5000, 8)
Bed capacity shape: (20, 3)

Admissions columns:
['Admission_ID', 'Patient_ID', 'Doctor_ID', 'Department_ID', 'Admission_Date', 'Discharge_Date', 'Status', 'Length_of_Stay_Days']

Bed capacity:


,department_id,department_name,Total_Beds
0,D001,Cardiology,40
1,D002,Neurology,30
2,D003,Orthopedics,35
3,D004,Pediatrics,45
4,D005,Oncology,35


In [3]:
print(admissions.columns.tolist())
display(admissions.head())

['Admission_ID', 'Patient_ID', 'Doctor_ID', 'Department_ID', 'Admission_Date', 'Discharge_Date', 'Status', 'Length_of_Stay_Days']


,Admission_ID,Patient_ID,Doctor_ID,Department_ID,Admission_Date,Discharge_Date,Status,Length_of_Stay_Days
0,A00001,P00001,DR00186,D002,2025-10-20,2025-10-28,Critical,8
1,A00002,P00002,DR00281,D019,2025-01-21,2025-01-23,Recovered,2
2,A00003,P00003,DR00105,D020,2025-06-24,2025-07-03,Discharged,9
3,A00004,P00004,DR00312,D004,2025-07-23,2025-07-25,Critical,2
4,A00005,P00005,DR00430,D018,2025-04-06,2025-04-07,Under Treatment,1


In [5]:
# Count actual admissions for each department
admission_counts = (
    admissions.groupby("Department_ID")
    .size()
    .reset_index(name="Actual_Admissions")
)

# Rename the admissions department column to match bed_capacity
admission_counts = admission_counts.rename(
    columns={"Department_ID": "department_id"}
)

# Merge actual admissions with benchmark bed capacity
capacity_analysis = bed_capacity.merge(
    admission_counts,
    on="department_id",
    how="left"
)

# Departments with no admissions will get 0
capacity_analysis["Actual_Admissions"] = (
    capacity_analysis["Actual_Admissions"].fillna(0)
)

# Calculate utilization rate
capacity_analysis["Utilization_Rate_%"] = (
    capacity_analysis["Actual_Admissions"]
    / capacity_analysis["Total_Beds"]
    * 100
)

# Calculate capacity gap
capacity_analysis["Capacity_Gap"] = (
    capacity_analysis["Actual_Admissions"]
    - capacity_analysis["Total_Beds"]
)

display(capacity_analysis)

,department_id,department_name,Total_Beds,Actual_Admissions,Utilization_Rate_%,Capacity_Gap
0,D001,Cardiology,40,233,582.500000,193
1,D002,Neurology,30,200,666.666667,170
2,D003,Orthopedics,35,320,914.285714,285
3,D004,Pediatrics,45,277,615.555556,232
4,D005,Oncology,35,331,945.714286,296
5,D006,ENT,15,215,1433.333333,200
6,D007,Dermatology,10,179,1790.000000,169
7,D008,General Surgery,40,175,437.500000,135
8,D009,Urology,20,169,845.000000,149
9,D010,Nephrology,25,218,872.000000,193


In [6]:
def classify_capacity(gap):
    if gap > 0:
        return "Overloaded"
    elif gap == 0:
        return "At Capacity"
    else:
        return "Underutilized"

capacity_analysis["Capacity_Status"] = (
    capacity_analysis["Capacity_Gap"].apply(classify_capacity)
)

display(capacity_analysis)

,department_id,department_name,Total_Beds,Actual_Admissions,Utilization_Rate_%,Capacity_Gap,Capacity_Status
0,D001,Cardiology,40,233,582.500000,193,Overloaded
1,D002,Neurology,30,200,666.666667,170,Overloaded
2,D003,Orthopedics,35,320,914.285714,285,Overloaded
3,D004,Pediatrics,45,277,615.555556,232,Overloaded
4,D005,Oncology,35,331,945.714286,296,Overloaded
5,D006,ENT,15,215,1433.333333,200,Overloaded
6,D007,Dermatology,10,179,1790.000000,169,Overloaded
7,D008,General Surgery,40,175,437.500000,135,Overloaded
8,D009,Urology,20,169,845.000000,149,Overloaded
9,D010,Nephrology,25,218,872.000000,193,Overloaded


In [7]:
output_path = "../data/processed/department_capacity_gap_analysis.csv"

capacity_analysis.to_csv(output_path, index=False)

print("Saved successfully:")
print(output_path)

Saved successfully:
../data/processed/department_capacity_gap_analysis.csv


In [8]:
def classify_capacity(gap):
    if gap > 0:
        return "Overloaded"
    elif gap == 0:
        return "At Capacity"
    else:
        return "Underutilized"

capacity_analysis["Capacity_Status"] = (
    capacity_analysis["Capacity_Gap"].apply(classify_capacity)
)

display(capacity_analysis)

,department_id,department_name,Total_Beds,Actual_Admissions,Utilization_Rate_%,Capacity_Gap,Capacity_Status
0,D001,Cardiology,40,233,582.500000,193,Overloaded
1,D002,Neurology,30,200,666.666667,170,Overloaded
2,D003,Orthopedics,35,320,914.285714,285,Overloaded
3,D004,Pediatrics,45,277,615.555556,232,Overloaded
4,D005,Oncology,35,331,945.714286,296,Overloaded
5,D006,ENT,15,215,1433.333333,200,Overloaded
6,D007,Dermatology,10,179,1790.000000,169,Overloaded
7,D008,General Surgery,40,175,437.500000,135,Overloaded
8,D009,Urology,20,169,845.000000,149,Overloaded
9,D010,Nephrology,25,218,872.000000,193,Overloaded


In [9]:
capacity_analysis[
    [
        "department_name",
        "Total_Beds",
        "Actual_Admissions",
        "Utilization_Rate_%",
        "Capacity_Gap",
        "Capacity_Status"
    ]
]

,department_name,Total_Beds,Actual_Admissions,Utilization_Rate_%,Capacity_Gap,Capacity_Status
0,Cardiology,40,233,582.500000,193,Overloaded
1,Neurology,30,200,666.666667,170,Overloaded
2,Orthopedics,35,320,914.285714,285,Overloaded
3,Pediatrics,45,277,615.555556,232,Overloaded
4,Oncology,35,331,945.714286,296,Overloaded
5,ENT,15,215,1433.333333,200,Overloaded
6,Dermatology,10,179,1790.000000,169,Overloaded
7,General Surgery,40,175,437.500000,135,Overloaded
8,Urology,20,169,845.000000,149,Overloaded
9,Nephrology,25,218,872.000000,193,Overloaded


In [10]:
# Summary KPIs

avg_utilization = capacity_analysis["Utilization_Rate_%"].mean()

most_overloaded = capacity_analysis.loc[
    capacity_analysis["Capacity_Gap"].idxmax()
]

largest_utilization = capacity_analysis.loc[
    capacity_analysis["Utilization_Rate_%"].idxmax()
]

total_beds = capacity_analysis["Total_Beds"].sum()
total_admissions = capacity_analysis["Actual_Admissions"].sum()
total_capacity_gap = capacity_analysis["Capacity_Gap"].sum()

print(f"Average Utilization: {avg_utilization:.2f}%")
print(f"Total Benchmark Beds: {total_beds}")
print(f"Total Admissions: {total_admissions}")
print(f"Total Capacity Gap: {total_capacity_gap}")
print(
    f"Department with Largest Capacity Gap: "
    f"{most_overloaded['department_name']} "
    f"({most_overloaded['Capacity_Gap']:.0f})"
)
print(
    f"Highest Utilization: "
    f"{largest_utilization['department_name']} "
    f"({largest_utilization['Utilization_Rate_%']:.2f}%)"
)

Average Utilization: 1160.62%
Total Benchmark Beds: 560
Total Admissions: 5000
Total Capacity Gap: 4440
Department with Largest Capacity Gap: Endocrinology (301)
Highest Utilization: Radiology (2660.00%)


In [17]:
import plotly.express as px

fig = px.bar(
    capacity_analysis,
    x="department_name",
    y=["Total_Beds", "Actual_Admissions"],
    barmode="group",
    title="Benchmark Capacity vs Actual Admissions by Department",
    labels={
        "department_name": "Department",
        "value": "Count",
        "variable": "Measure"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=600
)

fig.show()

In [15]:
import sys
print(sys.executable)

c:\Users\Deepika J\AppData\Local\Programs\Python\Python311\python.exe


In [16]:
import plotly
print(plotly.__version__)

7.0.0


In [18]:
# Sort departments by capacity gap
gap_sorted = capacity_analysis.sort_values(
    "Capacity_Gap",
    ascending=False
)

fig = px.bar(
    gap_sorted,
    x="department_name",
    y="Capacity_Gap",
    title="Capacity Gap by Department",
    labels={
        "department_name": "Department",
        "Capacity_Gap": "Capacity Gap"
    }
)

fig.update_layout(
    xaxis_tickangle=-45,
    height=600
)

fig.show()

In [19]:
# Benchmark & Capacity Gap KPIs

total_departments = capacity_analysis["department_name"].nunique()

overloaded_departments = (
    capacity_analysis["Capacity_Status"] == "Overloaded"
).sum()

at_capacity_departments = (
    capacity_analysis["Capacity_Status"] == "At Capacity"
).sum()

underutilized_departments = (
    capacity_analysis["Capacity_Status"] == "Underutilized"
).sum()

avg_utilization = capacity_analysis["Utilization_Rate_%"].mean()

highest_utilization_dept = capacity_analysis.loc[
    capacity_analysis["Utilization_Rate_%"].idxmax(),
    "department_name"
]

highest_utilization = capacity_analysis["Utilization_Rate_%"].max()

largest_gap_dept = capacity_analysis.loc[
    capacity_analysis["Capacity_Gap"].idxmax(),
    "department_name"
]

largest_gap = capacity_analysis["Capacity_Gap"].max()

print("BENCHMARK & CAPACITY GAP KPIs")
print("--------------------------------")
print(f"Total Departments: {total_departments}")
print(f"Overloaded Departments: {overloaded_departments}")
print(f"At Capacity Departments: {at_capacity_departments}")
print(f"Underutilized Departments: {underutilized_departments}")
print(f"Average Utilization: {avg_utilization:.2f}%")
print(
    f"Highest Utilization: {highest_utilization_dept} "
    f"({highest_utilization:.2f}%)"
)
print(
    f"Largest Capacity Gap: {largest_gap_dept} "
    f"({largest_gap:.0f})"
)

BENCHMARK & CAPACITY GAP KPIs
--------------------------------
Total Departments: 20
Overloaded Departments: 20
At Capacity Departments: 0
Underutilized Departments: 0
Average Utilization: 1160.62%
Highest Utilization: Radiology (2660.00%)
Largest Capacity Gap: Endocrinology (301)


In [20]:
kpi_table = pd.DataFrame({
    "KPI": [
        "Total Departments",
        "Overloaded Departments",
        "At Capacity Departments",
        "Underutilized Departments",
        "Average Utilization (%)",
        "Highest Utilization Department",
        "Highest Utilization (%)",
        "Largest Capacity Gap Department",
        "Largest Capacity Gap"
    ],
    "Value": [
        total_departments,
        overloaded_departments,
        at_capacity_departments,
        underutilized_departments,
        round(avg_utilization, 2),
        highest_utilization_dept,
        round(highest_utilization, 2),
        largest_gap_dept,
        largest_gap
    ]
})

display(kpi_table)

,KPI,Value
0,Total Departments,20
1,Overloaded Departments,20
2,At Capacity Departments,0
3,Underutilized Departments,0
4,Average Utilization (%),1160.62
5,Highest Utilization Department,Radiology
6,Highest Utilization (%),2660.0
7,Largest Capacity Gap Department,Endocrinology
8,Largest Capacity Gap,301


In [21]:
capacity_analysis.to_csv(
    "../data/processed/department_capacity_gap_analysis.csv",
    index=False
)

print("Final analysis file saved successfully.")

Final analysis file saved successfully.


In [22]:
status_counts = (
    capacity_analysis["Capacity_Status"]
    .value_counts()
    .reset_index()
)

status_counts.columns = ["Capacity_Status", "Department_Count"]

fig = px.bar(
    status_counts,
    x="Capacity_Status",
    y="Department_Count",
    title="Department Capacity Status",
    labels={
        "Capacity_Status": "Capacity Status",
        "Department_Count": "Number of Departments"
    },
    text="Department_Count"
)

fig.update_layout(height=500)

fig.show()

In [23]:
capacity_analysis.to_csv(
    "../data/processed/department_capacity_gap_analysis.csv",
    index=False
)

kpi_table.to_csv(
    "../data/processed/benchmark_capacity_kpis.csv",
    index=False
)

print("All analysis outputs saved successfully.")

All analysis outputs saved successfully.
